# LangGraph prompt chaining: product description pipeline
A four-node `StateGraph` where each node is one LLM call and passes its output to the next through typed state:

`START → basic description → features & benefits → marketing message → polished final description → END`

Uses any OpenRouter model through the OpenAI-compatible API; the key is read with `getpass`, never stored in the notebook.

In [1]:
!pip install -U langgraph langchain langchain-core langchain-community \
               langchain-openai langchain-experimental \
               wikipedia google-search-results \
               pydantic typing-extensions

In [2]:
# -----------------------------------------
# Imports
# -----------------------------------------
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
import os

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

In [3]:
import os
import getpass

os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API Key: ")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


Enter OpenRouter API Key: ··········


In [4]:
llm = ChatOpenAI(
    model="openrouter/free",
    temperature=0,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

class State(TypedDict):
    product_name: str
    basic_description: str
    features_benefits: str
    marketing_message: str
    final_description: str


In [6]:
def generate_basic_description(state: State):
  response = llm.invoke([
      SystemMessage(content="You are a helpful assistant that generates brief product descriptions."),
      HumanMessage(content=f"Write a brief description of a product named '{state['product_name']}'.")
  ])
  return {"basic_description": response.content}

def add_features_benefits(state: State):
  response = llm.invoke([
        HumanMessage(content=f"List key features and benefits of the product:\n{state['basic_description']}")
    ])
  return {"features_benefits": response.content}

def create_marketing_message(state: State):
  response = llm.invoke([
        HumanMessage(content=f"Create a compelling marketing message using:\n{state['features_benefits']}")
    ])
  return {"marketing_message": response.content}

def polish_final_description(state: State):
    response = llm.invoke([
        HumanMessage(content=f"""
Polish and finalize the product description using the marketing message below:

{state['marketing_message']}
""")
    ])
    return {"final_description": response.content}



In [ ]:
def build_workflow():
  workflow = StateGraph(State)

  workflow.add_node("basic", generate_basic_description)
  workflow.add_node("features", add_features_benefits)
  workflow.add_node("marketing", create_marketing_message)
  workflow.add_node("final", polish_final_description)

  workflow.add_edge(START, "basic")
  workflow.add_edge("basic", "features")
  workflow.add_edge("features", "marketing")
  workflow.add_edge("marketing", "final")

  return workflow.compile()


app = build_workflow()

In [ ]:
result = app.invoke({"product_name": "EcoBrew smart coffee mug"})

for step in ["basic_description", "features_benefits", "marketing_message", "final_description"]:
    print(f"\n===== {step} =====")
    print(result[step])